In [4]:
import pandas as pd
import os

# Create folder locally inside your current active project directory
os.makedirs("data/Feature engineered data", exist_ok=True)


# ---------------------------------------------------------
# 1. LOAD CLEANED DATA
# ---------------------------------------------------------
orders = pd.read_csv("C:/Users/Sakthi/OneDrive/Documents/Bharathi - Guvi-AIML/project final 1/Cart2Insight/data/Cleaned data/orders_clean.csv", parse_dates=[
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
])

order_items = pd.read_csv("C:/Users/Sakthi/OneDrive/Documents/Bharathi - Guvi-AIML/project final 1/Cart2Insight/data/Cleaned data/order_items_clean.csv")
order_reviews = pd.read_csv("C:/Users/Sakthi/OneDrive/Documents/Bharathi - Guvi-AIML/project final 1/Cart2Insight/data/Cleaned data/order_reviews_clean.csv")
customers = pd.read_csv("C:/Users/Sakthi/OneDrive/Documents/Bharathi - Guvi-AIML/project final 1/Cart2Insight/data/Cleaned data/customers_clean.csv")
sellers = pd.read_csv("C:/Users/Sakthi/OneDrive/Documents/Bharathi - Guvi-AIML/project final 1/Cart2Insight/data/Cleaned data/sellers_clean.csv")

print("Loaded cleaned datasets for feature engineering!")

# ---------------------------------------------------------
# 2. ORDER VALUE (price + freight_value)
# ---------------------------------------------------------
order_items["item_total"] = order_items["price"] + order_items["freight_value"]

order_value = order_items.groupby("order_id")["item_total"].sum().reset_index()
order_value = order_value.rename(columns={"item_total": "order_value"})

orders = orders.merge(order_value, on="order_id", how="left")
orders["order_value"] = orders["order_value"].fillna(0)

print("Order value feature created!")

# ---------------------------------------------------------
# 3. DELIVERY DAYS & DELIVERY DELAY
# ---------------------------------------------------------
orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days

print("Delivery features created!")

# ---------------------------------------------------------
# 4. CUSTOMER SPENDING & TIER (FIXED: Grouping by unique lifetime ID)
# ---------------------------------------------------------
# Step A: Link customer_unique_id to the orders dataframe
orders_with_unique_id = orders.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left")

# Step B: Group by lifetime unique ID instead of order-level customer_id
customer_spend = orders_with_unique_id.groupby("customer_unique_id")["order_value"].sum().reset_index()
customer_spend = customer_spend.rename(columns={"order_value": "customer_total_spent"})

# Step C: Merge lifetime spending back into the master customers table
customers = customers.merge(customer_spend, on="customer_unique_id", how="left")
customers["customer_total_spent"] = customers["customer_total_spent"].fillna(0)

# Spending tiers (Low, Medium, High)
q1 = customers["customer_total_spent"].quantile(0.33)
q2 = customers["customer_total_spent"].quantile(0.66)

def spending_tier(x):
    if x <= q1:
        return "Low"
    elif x <= q2:
        return "Medium"
    else:
        return "High"

customers["customer_spending_tier"] = customers["customer_total_spent"].apply(spending_tier)

print("Customer spending tiers created!")

# ---------------------------------------------------------
# 5. SELLER PERFORMANCE FEATURES
# ---------------------------------------------------------
# Seller revenue
seller_revenue = order_items.groupby("seller_id")["item_total"].sum().reset_index()
seller_revenue = seller_revenue.rename(columns={"item_total": "seller_revenue"})

# Seller order count
seller_order_count = order_items.groupby("seller_id")["order_id"].nunique().reset_index()
seller_order_count = seller_order_count.rename(columns={"order_id": "seller_order_count"})

# Seller average review score
seller_reviews = order_items.merge(
    order_reviews[["order_id", "review_score"]],
    on="order_id",
    how="left"
)

seller_avg_review = seller_reviews.groupby("seller_id")["review_score"].mean().reset_index()
seller_avg_review = seller_avg_review.rename(columns={"review_score": "seller_avg_review_score"})

# Merge into sellers table
sellers = sellers.merge(seller_revenue, on="seller_id", how="left")
sellers = sellers.merge(seller_order_count, on="seller_id", how="left")
sellers = sellers.merge(seller_avg_review, on="seller_id", how="left")

print("Seller performance features created!")

# ---------------------------------------------------------
# 6. SAVE FEATURE ENGINEERED DATASETS
# ---------------------------------------------------------
orders.to_csv("C:/Users/Sakthi/OneDrive/Documents/Bharathi - Guvi-AIML/project final 1/Cart2Insight/data/Feature engineered data/orders_fe.csv", index=False)
customers.to_csv("C:/Users/Sakthi/OneDrive/Documents/Bharathi - Guvi-AIML/project final 1/Cart2Insight/data/Feature engineered data/customers_fe.csv", index=False)
sellers.to_csv("C:/Users/Sakthi/OneDrive/Documents/Bharathi - Guvi-AIML/project final 1/Cart2Insight/data/Feature engineered data/sellers_fe.csv", index=False)

print("Feature-engineered datasets saved!")


Loaded cleaned datasets for feature engineering!
Order value feature created!
Delivery features created!
Customer spending tiers created!
Seller performance features created!
Feature-engineered datasets saved!
